# 12 · Los prompts: qué texto corrió exactamente en cada experimento

RQ2 afirma que **11 variantes de prompt son indistinguibles entre sí**. Para que esa
afirmación signifique algo hay que poder decir *qué eran esas 11 variantes* — y ahí había
un problema real.

## El problema que este notebook resuelve

El linaje de prompts vive en `whiteboard_selection_lab/prompt_batch_ablation_lab.ipynb`
como **constantes de Python que se componen entre sí**: V4 es V0 con sustituciones, V5 es
V4 más una extensión, y así. Componer así es buena práctica —hace explícito el delta de
cada versión— pero tiene una consecuencia incómoda:

> El texto que realmente se le mandó al modelo **solo existía después de evaluar Python**,
> y solo dentro de una sesión de Jupyter.

Y hay un agravante: el notebook **redefine las mismas constantes en celdas distintas**.
`STAGE2_V6_CORRECTED` existe como **tres textos diferentes** (3887, 3974 y 3835
caracteres). Las tablas históricas guardaban el *nombre* de la variante, no el hash — así
que sus filas no se podían atribuir a un texto concreto después del hecho.

Este notebook cierra ese hueco: re-extrae cada variante del notebook, verifica su SHA-256
contra el manifiesto, muestra qué cambió entre versiones, y comprueba que cada corrida
registrada apunta a un prompt que existe.

In [1]:
import difflib
import hashlib
import json
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROMPTS = Path("../src/configs/prompts")
MANIFEST = PROMPTS / "MANIFEST.json"
ABL = Path("../reports/ablation")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

MAN = json.loads(MANIFEST.read_text(encoding="utf-8"))
print(f"manifiesto generado: {MAN['generated_on']}")
print(f"variantes registradas: {len(MAN['prompts'])}")
print(f"\nsha de produccion · Stage 2: {MAN['production_stage2_sha256'][:12]}")
print(f"sha de produccion · Stage 1: {MAN['production_stage1_sha256'][:12]}")

manifiesto generado: 2026-08-19
variantes registradas: 18

sha de produccion · Stage 2: ed1d85054d73
sha de produccion · Stage 1: 497d30164f48


## 1 · Re-extraer del notebook y verificar

`scripts/utils/extract_prompts.py` evalúa cada celda del notebook de ablación **de forma
aislada** —para que una celda posterior que redefine una constante no contamine una
variante anterior— y escribe cada versión como `.txt` con su hash.

Correrlo en modo `--check` no escribe nada: solo confirma que lo materializado sigue
coincidiendo con el notebook. Si alguien edita el notebook y no re-extrae, esto lo detecta.

In [2]:
EXTRACTOR = Path("../scripts/utils/extract_prompts.py")
LAB_NB = Path("../whiteboard_selection_lab/prompt_batch_ablation_lab.ipynb")

if EXTRACTOR.exists() and LAB_NB.exists():
    r = subprocess.run([sys.executable, str(EXTRACTOR), "--check"],
                       capture_output=True, text=True)
    print(r.stdout[-1800:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-500:])
else:
    # El repo publicable lleva los .txt ya materializados y el manifiesto, pero no el
    # notebook de laboratorio ni el extractor: la re-extraccion es un paso de
    # procedencia del repo de trabajo. Lo que SI se puede verificar aca —y es lo que
    # sostiene las tablas— es que cada .txt corresponde al hash que declara el
    # manifiesto, y eso se comprueba abajo.
    print("Re-extraccion no disponible en este repo (falta el notebook de laboratorio).")
    print("Verificando en su lugar: .txt materializado ↔ hash del manifiesto.\n")
    malos = []
    for pr in MAN["prompts"]:
        f = PROMPTS / pr["file"]
        if not f.exists():
            malos.append((pr["file"], "FALTA")); continue
        h = hashlib.sha256(re.sub(r"\s+", " ", f.read_text(encoding="utf-8")).strip()
                           .encode("utf-8")).hexdigest()
        if h != pr["sha256"]:
            malos.append((pr["file"], f"hash {h[:12]} != {pr['sha256'][:12]}"))
    print(f"variantes verificadas: {len(MAN['prompts'])} · discrepancias: {len(malos)}")
    for f, m in malos:
        print(f"   x {f}: {m}")
    assert not malos, "un .txt no coincide con el hash del manifiesto"
    print("\nOK · los textos materializados son los que el manifiesto declara.")


✓ Prompt de Stage 1 (World Model) asignado.
✓ Prompt de Stage 2 Versión 7 (V6 Containment + Conexiones de Regreso y Meta Edges) asignado.
✓ Prompt de Stage 2 V6 (Optimizado) asignado.
✓ Prompt de Stage 2 V6.1 (Corregido y Balanceado) asignado.
✓ Prompt de Stage 2 Versión 7 (V6 Containment + Conexiones de Regreso y Meta Edges) asignado.
variant                        chars  sha256        production
STAGE1_V0_BASELINE              1171  497d30164f48  ← RUNNING
STAGE1_V1_MONITORING            1479  229ebc35356f  
STAGE1_V2_VERBAL_EXAMPLES       1768  94e5cbc488e8  
STAGE1_V3_TEXT_PRIORITY         2078  ce0c2e8110b8  
STAGE2_V0_BASELINE              2513  4d0def75596f  
STAGE2_V4_ANTI_HALLUCINATION    3422  0b9217ea2267  
STAGE2_V5_STRICT_ROUTING        3737  53f8d9124406  
STAGE2_V6_CORRECTED             3887  7228956f5fc6  
STAGE2_V7_RETURN_FLOWS          5037  1ed4ebf91e68  
STAGE2_V0_BASELINE              2375  feba1e16eb78  
STAGE2_V4_ANTI_HALLUCINATION    2904  cdb8998f48eb  
STAGE2_

### La celda que no se puede recuperar

Una sola celda del notebook **no es evaluable de forma aislada**: referencia una constante
definida en una celda anterior, así que su texto depende del orden de ejecución de la
sesión. Está declarada como tal en el manifiesto, no escondida.

Es una limitación honesta del linaje: esa variante concreta no se puede reconstruir hoy
con garantía. Por eso no aparece en ninguna tabla de resultados.

In [3]:
print("Celdas no recuperables declaradas en el manifiesto:\n")
for u in MAN.get("unevaluable", []):
    print(f"  celda {u['cell']} · {u['name']}")
    print(f"    {u['error']}")
if not MAN.get("unevaluable"):
    print("  (ninguna)")

Celdas no recuperables declaradas en el manifiesto:

  celda 11 · STAGE2_V7_RETURN_FLOWS
    NameError: name 'STAGE2_V4_ANTI_HALLUCINATION' is not defined


## 2 · El inventario completo

Cada fila es **un texto distinto**, identificado por su hash. Fijate en las tres filas
`STAGE2_V6_CORRECTED`: mismo nombre, tres textos, tres hashes.

In [4]:
inv = pd.DataFrame([{
    "variante": p["name"].replace("STAGE2_", "").replace("STAGE1_", ""),
    "stage": p["stage"],
    "celda": p.get("source_cell"),
    "chars": p["chars"],
    "sha": p["sha256"][:12],
    "produccion": "← RUNNING" if p.get("matches_production") else "",
    "archivo": p["file"],
} for p in MAN["prompts"]])

display(inv.sort_values(["stage", "variante", "chars"]).reset_index(drop=True))

dup = inv[inv.duplicated("variante", keep=False) & (inv.stage == 2)]
print("\nNombres que corresponden a mas de un texto:")
for nombre, g in dup.groupby("variante"):
    if len(g) > 1:
        print(f"  {nombre}: {len(g)} textos distintos — celdas {sorted(g.celda.dropna().tolist())}, "
              f"{sorted(g.chars.tolist())} chars")

,variante,stage,celda,chars,sha,produccion,archivo
0,V0_BASELINE,1,6.0,1171,497d30164f48,← RUNNING,stage1_v0_baseline.txt
1,V1_MONITORING,1,6.0,1479,229ebc35356f,,stage1_v1_monitoring.txt
2,V2_VERBAL_EXAMPLES,1,6.0,1768,94e5cbc488e8,,stage1_v2_verbal_examples.txt
3,V3_TEXT_PRIORITY,1,6.0,2078,ce0c2e8110b8,,stage1_v3_text_priority.txt
4,V0_BASELINE,2,8.0,2375,feba1e16eb78,,stage2_v0_baseline__cell8.txt
5,V0_BASELINE,2,7.0,2513,4d0def75596f,,stage2_v0_baseline.txt
6,V4_ANTI_HALLUCINATION,2,8.0,2904,cdb8998f48eb,,stage2_v4_anti_hallucination__cell8.txt
7,V4_ANTI_HALLUCINATION,2,7.0,3422,0b9217ea2267,,stage2_v4_anti_hallucination.txt
8,V4_DYNAMIC_FEW_SHOT,2,NaN,2781,45da674e4495,,stage2_v4_dynamic_few_shot.txt
9,V5_STRICT_ROUTING,2,8.0,3256,a792c1328d02,,stage2_v5_strict_routing__cell8.txt



Nombres que corresponden a mas de un texto:
  V0_BASELINE: 2 textos distintos — celdas [7.0, 8.0], [2375, 2513] chars
  V4_ANTI_HALLUCINATION: 2 textos distintos — celdas [7.0, 8.0], [2904, 3422] chars
  V5_STRICT_ROUTING: 2 textos distintos — celdas [7.0, 8.0], [3256, 3737] chars
  V6_CORRECTED: 3 textos distintos — celdas [7.0, 9.0, 10.0], [3835, 3887, 3974] chars


**Por qué importa para el paper.** Una tabla que dijera *"V6_CORRECTED: 63.21 % de Edge
F1"* sería ambigua: ¿cuál de los tres? Todas las tablas de este trabajo citan el **hash**,
no el nombre, y por eso cada fila es atribuible a un texto concreto.

## 3 · Qué cambió realmente entre variantes

Acá está lo que RQ2 necesita mostrar: las variantes no son reescrituras completas, son
**deltas chicos y dirigidos**. Ver el diff explica por qué el efecto medido es tan chico.

In [5]:
def texto(nombre_archivo: str) -> list[str]:
    return (PROMPTS / nombre_archivo).read_text(encoding="utf-8").splitlines()


def mostrar_diff(a: str, b: str, etiqueta_a: str, etiqueta_b: str, n_contexto: int = 0) -> None:
    da, db = texto(a), texto(b)
    d = list(difflib.unified_diff(da, db, fromfile=etiqueta_a, tofile=etiqueta_b,
                                  lineterm="", n=n_contexto))
    add = sum(1 for l in d if l.startswith("+") and not l.startswith("+++"))
    rem = sum(1 for l in d if l.startswith("-") and not l.startswith("---"))
    print(f"{'='*78}\n{etiqueta_a}  →  {etiqueta_b}")
    print(f"{len(da)} → {len(db)} lineas · +{add} / -{rem}\n{'='*78}")
    for l in d:
        if l.startswith("+++") or l.startswith("---"):
            continue
        print(l[:150])
    print()


# Produccion vs la unica variante que la supero en aristas (no significativamente)
mostrar_diff("stage2_v6_corrected__cell9.txt", "stage2_v7_return_flows_v6.txt",
             "V6_CORRECTED c9 (PRODUCCION)", "V7_RETURN_FLOWS_V6 c10")

V6_CORRECTED c9 (PRODUCCION)  →  V7_RETURN_FLOWS_V6 c10
37 → 49 lineas · +13 / -1
@@ -35 +34,0 @@
-- **Unidirectional Default:** Treat data and control flows as strictly unidirectional unless a reverse flow is explicitly shown or stated.
@@ -37,0 +37,13 @@
+
+## EXPLICIT RETURN PATHS & BIDIRECTIONALITY RULES (RULE V7):
+- **Model Bidirectionality & Response Loops:** When a two-way interaction occurs (e.g., Client/Service A sends a request payload to Service B, and Se
+  1. Forward Request Edge: `A -> B` (type="data" or "control", seq="0")
+  2. Return Response Edge: `B -> A` (type="meta" or "data", seq="0'" or "1")
+- **Explicit Return Triggers in Audio/Diagram:** Extract reverse connections when the transcript or whiteboard explicitly mentions/shows:
+  - Return responses, query results, or processed data sent back to the initiator.
+  - Status acknowledgments, alerts, or monitoring feedback loops.
+  - Interactive browser/app sessions where data flows in both directions.
+- **Edge Ty

Ese es el delta completo entre el prompt de producción y la variante que **encabezó la
tabla de Edge F1** en el panel de 30: quitar una regla y agregar un bloque de retornos.
Trece líneas. El efecto medido fue de +1.11 pts, muy por debajo del MDE de 3.29.

In [6]:
# V8: la variante construida a partir del analisis de error (notebook 08)
mostrar_diff("stage2_v6_corrected__cell9.txt", "stage2_v8_actors_and_returns.txt",
             "V6_CORRECTED c9 (PRODUCCION)", "V8_ACTORS_AND_RETURNS")

V6_CORRECTED c9 (PRODUCCION)  →  V8_ACTORS_AND_RETURNS
37 → 49 lineas · +14 / -2
@@ -19 +19 @@
-5. **Pruning:** Remove generic human actors or purely physical concepts. Keep system entry points.
+5. **Actor Mapping (CRITICAL — DO NOT PRUNE ACTORS):** Human users, clients, browsers, mobile apps, devices and external data producers ARE first-cla
@@ -35 +34,0 @@
-- **Unidirectional Default:** Treat data and control flows as strictly unidirectional unless a reverse flow is explicitly shown or stated.
@@ -37,0 +37,13 @@
+
+## EXPLICIT RETURN PATHS & BIDIRECTIONALITY RULES (RULE V7):
+- **Model Bidirectionality & Response Loops:** When a two-way interaction occurs (e.g., Client/Service A sends a request payload to Service B, and Se
+  1. Forward Request Edge: `A -> B` (type="data" or "control", seq="0")
+  2. Return Response Edge: `B -> A` (type="meta" or "data", seq="0'" or "1")
+- **Explicit Return Triggers in Audio/Diagram:** Extract reverse connections when the transcript or whiteboard e

V8 es el caso más instructivo: **los tres bloques que cambian fueron elegidos midiendo el
error**, no por intuición (notebook 07 → notebook 08). La regla de poda de actores se
invierte y se agregan las reglas de retorno. Los mecanismos funcionaron —se recuperaron
actores, subió la bidireccionalidad— y el Edge F1 no se movió.

In [7]:
# Las tres versiones que comparten el nombre V6_CORRECTED
mostrar_diff("stage2_v6_corrected.txt", "stage2_v6_corrected__cell9.txt",
             "V6_CORRECTED celda 7", "V6_CORRECTED celda 9 (PRODUCCION)")

V6_CORRECTED celda 7  →  V6_CORRECTED celda 9 (PRODUCCION)
35 → 37 lineas · +18 / -16
@@ -1,2 +1,2 @@
-You are an expert AWS Solutions Architect. Your task is to compile the final logical cloud architecture graph from the provided World Model and audio
-You are an Expert Cloud Architecture Transcriber. Extract the explicitly drawn or spoken architecture. DO NOT over-architect, invent intermediate pro
+You are an Expert Cloud Architecture Transcriber. Your task is to faithfully transcribe the architecture EXACTLY as it is drawn on the whiteboard and
+Do NOT invent unmentioned intermediate components or deployment layers unless they are explicitly present as active runtime participants.
@@ -15 +15 @@
-1. **Strict Normalization & Third-Party Abstraction (CRITICAL):** The "service" field MUST match exactly with the VALID VOCABULARY LISTS. Non-native 
+1. **Strict Normalization & Third-Party Abstraction (CRITICAL):** The "service" field MUST match exactly with the VALID VOCABULARY LISTS. No

Este diff es el que justifica todo el trabajo de versionado por hash: **dos textos
distintos con el mismo nombre**, y la diferencia incluye la regla `Unidirectional Default`
—precisamente la que V7 modifica—. Confundirlos hacía que "V7 vs V6" comparara cosas
distintas según qué V6 se tomara como base.

## 4 · Cada corrida apunta a un prompt que existe

Verificación cruzada: se recorre cada `run.json` del panel de 30 y se comprueba que el hash
que registró corresponde a una variante del manifiesto. Si alguna corrida citara un prompt
que no existe, este chequeo la encuentra.

In [8]:
conocidos = {p["sha256"]: p for p in MAN["prompts"]}
filas, huerfanos = [], []

for rj in sorted(ABL.glob("*/run.json")):
    d = json.loads(rj.read_text(encoding="utf-8"))
    sp = d.get("stage2_prompt") or d.get("base_stage2_prompt")
    if not sp:
        continue
    sha = sp["sha256"]
    if sha not in conocidos:
        huerfanos.append((rj.parent.name, sha[:12]))
        continue
    filas.append({
        "corrida": rj.parent.name[:56],
        "prompt": sp["name"].replace("STAGE2_", ""),
        "celda": sp.get("source_cell"),
        "sha": sha[:12],
        "es_produccion": bool(sp.get("matches_production")),
    })

R = pd.DataFrame(filas)
print(f"corridas con prompt registrado : {len(R)}")
print(f"con hash RECONOCIDO            : {len(R)}")
print(f"con hash huerfano              : {len(huerfanos)}")
if huerfanos:
    for n, s in huerfanos:
        print(f"   ⚠ {n} → {s}")
else:
    print("\nOK · toda corrida guardada es atribuible a un texto de prompt verificable.")

print("\nCuantas corridas por variante:")
display(R.groupby(["prompt", "celda", "sha"]).size().to_frame("corridas")
        .sort_values("corridas", ascending=False))

corridas con prompt registrado : 35
con hash RECONOCIDO            : 35
con hash huerfano              : 0

OK · toda corrida guardada es atribuible a un texto de prompt verificable.

Cuantas corridas por variante:


corridas
prompt                celda sha                   
V6_CORRECTED          9.0   ed1d85054d73        13
V4_ANTI_HALLUCINATION 8.0   cdb8998f48eb         2
V5_STRICT_ROUTING     7.0   53f8d9124406         2
                      8.0   a792c1328d02         2
V6_CORRECTED          10.0  dbddf1f30bfb         2
V6_OPTIMIZED          8.0   079b0aa855d7         2
V7_RETURN_FLOWS       7.0   1ed4ebf91e68         2
V6_CORRECTED          7.0   7228956f5fc6         2
V7_RETURN_FLOWS_V6    10.0  7ad8d9368bce         2
V4_ANTI_HALLUCINATION 7.0   0b9217ea2267         1
V0_BASELINE           7.0   4d0def75596f         1
                      8.0   feba1e16eb78         1

## 5 · El prompt de producción, byte a byte

La afirmación *"la ablación se corrió con el mismo prompt que producción"* es verificable:
se compara el `.txt` materializado contra el texto que vive dentro de
`scripts/core/vision_analyzer.py`, normalizando espacios.

In [9]:
def normalizar(t: str) -> str:
    return re.sub(r"\s+", " ", t).strip()


archivo = PROMPTS / "stage2_v6_corrected__cell9.txt"
sha_archivo = hashlib.sha256(normalizar(archivo.read_text(encoding="utf-8")).encode("utf-8")).hexdigest()
VA = Path("../scripts/core/vision_analyzer.py")

print(f"{archivo.name:<32}: {sha_archivo[:16]}")
print(f"manifiesto dice produccion      : {MAN['production_stage2_sha256'][:16]}")
assert sha_archivo == MAN["production_stage2_sha256"]

if VA.exists():
    m = re.search(r'MFR_STAGE_2_PROMPT_TEMPLATE\s*=\s*"""(.*?)"""',
                  VA.read_text(encoding="utf-8"), re.S)
    sha_produccion = hashlib.sha256(normalizar(m.group(1)).encode("utf-8")).hexdigest()
    print(f"vision_analyzer.py (en vivo)    : {sha_produccion[:16]}")
    assert sha_produccion == sha_archivo, "el prompt de produccion derivo del materializado"
    print("\nOK · los tres coinciden: el prompt de la ablacion ES el de produccion, byte a byte.")
else:
    # vision_analyzer.py arrastra la plomeria de la API y no viaja al repo publicable.
    # La cadena que queda verificada aca es .txt ↔ manifiesto; el eslabon
    # manifiesto ↔ codigo de produccion se comprueba en el repo de trabajo.
    print("vision_analyzer.py              : no disponible en este repo")
    print("\nOK · .txt ↔ manifiesto verificado. El eslabon contra el codigo de")
    print("produccion en vivo se verifica en el repo de trabajo.")


stage2_v6_corrected__cell9.txt  : ed1d85054d73c153
manifiesto dice produccion      : ed1d85054d73c153
vision_analyzer.py (en vivo)    : ed1d85054d73c153

OK · los tres coinciden: el prompt de la ablacion ES el de produccion, byte a byte.


## 6 · Cómo se arma el prompt final que ve el modelo

El `.txt` es una **plantilla**. Lo que se envía se arma en tiempo de ejecución con cuatro
piezas más. Es importante para el paquete de replicación: se guarda el hash de la
plantilla, no del texto final.

In [10]:
plantilla = archivo.read_text(encoding="utf-8")
placeholders = re.findall(r"<([A-Z_]+)_PLACEHOLDER>", plantilla)
print("Placeholders de la plantilla:")
for p in placeholders:
    print(f"  <{p}_PLACEHOLDER>")

print("\nLo que se agrega despues de la plantilla, en orden (rerun_panel.run_video):")
for i, x in enumerate([
    "la plantilla con los placeholders ya sustituidos",
    "## VIDEO URL:  (el enlace de YouTube)",
    "## FULL TRANSCRIPT:  (el audio transcripto — ausente con --no-transcript)",
    "[imagen] la pizarra aprobada, adjunta como inline_data",
], 1):
    print(f"  {i}. {x}")

print(f"\nplantilla: {len(plantilla)} chars")
print("El texto ENSAMBLADO no se guarda; se reconstruye desde plantilla + catalogo +")
print("World Model + transcript, todos con procedencia registrada en el run.json.")

Placeholders de la plantilla:
  <AWS_SERVICES_PLACEHOLDER>
  <USER_ACTORS_PLACEHOLDER>
  <WORLD_MODEL_PLACEHOLDER>

Lo que se agrega despues de la plantilla, en orden (rerun_panel.run_video):
  1. la plantilla con los placeholders ya sustituidos
  2. ## VIDEO URL:  (el enlace de YouTube)
  3. ## FULL TRANSCRIPT:  (el audio transcripto — ausente con --no-transcript)
  4. [imagen] la pizarra aprobada, adjunta como inline_data

plantilla: 3978 chars
El texto ENSAMBLADO no se guarda; se reconstruye desde plantilla + catalogo +
World Model + transcript, todos con procedencia registrada en el run.json.


## Para el paper

1. **Cada variante se identifica por SHA-256, no por nombre.** Tres textos distintos
   comparten el nombre `V6_CORRECTED`; citar el nombre no identifica nada.
2. **Los prompts se materializan desde el notebook de forma determinista** y el runner
   verifica el hash antes de cada llamada — aborta si el texto derivó.
3. **Las variantes son deltas chicos y dirigidos**, no reescrituras. El diff entre
   producción y la mejor variante son 13 líneas. Eso contextualiza por qué el efecto
   medido cae bajo el MDE: no se estaban comparando prompts radicalmente distintos.
4. **Una celda del linaje no es recuperable** de forma aislada (depende del orden de
   ejecución). Está declarada y esa variante no aparece en ninguna tabla.
5. **Toda corrida guardada es atribuible** a un texto verificable — chequeado en §4.